# 04 - Analysis and Figures (Simple Upload Version)

This notebook combines your saved results and creates report-ready outputs.

It is written to be simple and Colab-friendly:

- manually upload the files you need
- minimal path logic
- saves all outputs automatically
- automatically downloads a zip of the saved outputs at the end

## Files you will upload
Required:
- `taskA_dataset.csv`
- `taskB_dataset.csv`
- `taskA_classical_results.csv`
- `taskB_classical_results.csv`
- `taskA_ffnn_results.csv`
- `taskB_ffnn_results.csv`

Optional for ROC curves:
- `taskA_preds_logistic_regression.csv`
- `taskA_preds_knn.csv`
- `taskA_preds_svm.csv`
- `taskA_preds_random_forest.csv`
- `taskA_preds_ffnn.csv`
- `taskB_preds_logistic_regression.csv`
- `taskB_preds_knn.csv`
- `taskB_preds_svm.csv`
- `taskB_preds_random_forest.csv`
- `taskB_preds_ffnn.csv`

## Files this notebook saves
- combined model-comparison tables
- betting-odds baseline table
- F1 comparison plot
- accuracy comparison plot
- Task B feature-importance plot
- Task B minimal-subset table and plot
- optional ROC plots if prediction files are uploaded
- one zip containing all analysis outputs


In [ ]:

# Basic imports
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss, roc_curve

pd.set_option('display.max_columns', 200)
sns.set_style("whitegrid")


## 1) Upload required files

In [ ]:

from google.colab import files

OUTPUT_DIR = '/content/ds4400_valorant_analysis_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Upload the required CSV files. You can also upload prediction files now if you have them.")
uploaded = files.upload()

required_files = [
    'taskA_dataset.csv',
    'taskB_dataset.csv',
    'taskA_classical_results.csv',
    'taskB_classical_results.csv',
    'taskA_ffnn_results.csv',
    'taskB_ffnn_results.csv'
]

missing_required = [f for f in required_files if f not in uploaded and not os.path.exists(f)]
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}")

taskA = pd.read_csv('taskA_dataset.csv', parse_dates=['Date'])
taskB = pd.read_csv('taskB_dataset.csv', parse_dates=['Date'])
taskA_classical = pd.read_csv('taskA_classical_results.csv')
taskB_classical = pd.read_csv('taskB_classical_results.csv')
taskA_ffnn = pd.read_csv('taskA_ffnn_results.csv')
taskB_ffnn = pd.read_csv('taskB_ffnn_results.csv')

print("Loaded required files successfully.")
print(taskA.shape, taskB.shape)


## 2) Combine results tables

In [ ]:

taskA_all = pd.concat([taskA_classical, taskA_ffnn], ignore_index=True)
taskB_all = pd.concat([taskB_classical, taskB_ffnn], ignore_index=True)
all_results = pd.concat([taskA_all, taskB_all], ignore_index=True)

taskA_all = taskA_all.sort_values('F1', ascending=False).reset_index(drop=True)
taskB_all = taskB_all.sort_values('F1', ascending=False).reset_index(drop=True)
all_results = all_results.sort_values(['Task', 'F1'], ascending=[True, False]).reset_index(drop=True)

display(taskA_all)
display(taskB_all)


## 3) Betting-odds baseline for Task B

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss

# Start from the Task B test set
baseline_df = taskB[taskB['Split'] == 'test'].copy()

# VLR odds are stored in decimal format (values greater than 1) on rows where odds exist,
# and encoded as 0 on rows where no market odds are available. We restrict the baseline to rows
# where a valid decimal odds quote exists, then convert to implied probability = 1 / odds.
odds_col = 'Team1 Map Odds' if 'Team1 Map Odds' in baseline_df.columns else 'Series Odds'

baseline_df = baseline_df[['Team1 Win', odds_col]].copy()
baseline_df = baseline_df.rename(columns={odds_col: 'DecimalOdds'})
baseline_df['DecimalOdds'] = pd.to_numeric(baseline_df['DecimalOdds'], errors='coerce')

# Keep only rows where the sportsbook actually provided odds (decimal odds are always greater than 1)
baseline_df = baseline_df[baseline_df['DecimalOdds'] > 1.0].copy()

# Convert decimal odds to implied win probability for Team1
baseline_df['ImpliedProb'] = 1.0 / baseline_df['DecimalOdds']

baseline_df['y_pred'] = (baseline_df['ImpliedProb'] >= 0.5).astype(int)

n_rows_used = len(baseline_df)
print(f"Odds baseline evaluated on {n_rows_used} test rows with valid market odds "
      f"(out of {(taskB['Split'] == 'test').sum()} total test rows).")

baseline_results_df = pd.DataFrame([{
    'Task': 'Task B',
    'Model': f'Odds Baseline ({odds_col})',
    'Accuracy': accuracy_score(baseline_df['Team1 Win'], baseline_df['y_pred']),
    'Precision': precision_score(baseline_df['Team1 Win'], baseline_df['y_pred'], zero_division=0),
    'Recall': recall_score(baseline_df['Team1 Win'], baseline_df['y_pred'], zero_division=0),
    'F1': f1_score(baseline_df['Team1 Win'], baseline_df['y_pred'], zero_division=0),
    'ROC_AUC': roc_auc_score(baseline_df['Team1 Win'], baseline_df['ImpliedProb']),
    'Brier': brier_score_loss(baseline_df['Team1 Win'], baseline_df['ImpliedProb'])
}])

baseline_results_df

## 4) Save comparison tables

In [ ]:

taskA_all_path = os.path.join(OUTPUT_DIR, 'taskA_all_results.csv')
taskB_all_path = os.path.join(OUTPUT_DIR, 'taskB_all_results.csv')
all_results_path = os.path.join(OUTPUT_DIR, 'all_model_results.csv')
betting_path = os.path.join(OUTPUT_DIR, 'taskB_betting_baseline.csv')

taskA_all.to_csv(taskA_all_path, index=False)
taskB_all.to_csv(taskB_all_path, index=False)
all_results.to_csv(all_results_path, index=False)
baseline_results_df.to_csv(betting_path, index=False)

print("Saved results tables.")


## 5) Plot model comparison figures

In [ ]:

def save_metric_barplot(df, metric, filename, title):
    plt.figure(figsize=(9, 5))
    order = df.sort_values(metric, ascending=False)['Model']
    sns.barplot(data=df, x='Model', y=metric, order=order)
    plt.xticks(rotation=30, ha='right')
    plt.title(title)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    return path

taskB_plot_df = pd.concat([taskB_all, baseline_results_df], ignore_index=True)

f1_plot_path = save_metric_barplot(taskB_plot_df, 'F1', 'taskB_f1_comparison.png', 'Task B F1 Comparison')
acc_plot_path = save_metric_barplot(taskB_plot_df, 'Accuracy', 'taskB_accuracy_comparison.png', 'Task B Accuracy Comparison')


## 6) Task B feature importance using Random Forest permutation importance

In [ ]:

def split_train_test(df, target_col='Team1 Win'):
    train_df = df[df['Split'] == 'train'].copy()
    test_df = df[df['Split'] == 'test'].copy()

    X_train = train_df.drop(columns=['Date', 'Split', target_col], errors='ignore')
    y_train = train_df[target_col].copy()
    X_test = test_df.drop(columns=['Date', 'Split', target_col], errors='ignore')
    y_test = test_df[target_col].copy()
    return X_train, X_test, y_train, y_test

def make_preprocessor(X):
    cat_cols = [c for c in X.columns if c == 'Map']
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_cols)
        ],
        remainder='drop'
    )
    return preprocessor

X_train_B, X_test_B, y_train_B, y_test_B = split_train_test(taskB)

pre_B = make_preprocessor(X_train_B)
rf_model = Pipeline([
    ('pre', pre_B),
    ('clf', RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train_B, y_train_B)

perm = permutation_importance(
    rf_model, X_test_B, y_test_B,
    n_repeats=5,
    random_state=42,
    scoring='f1',
    n_jobs=-1
)

feature_importance = pd.DataFrame({
    'Feature': X_test_B.columns,
    'Importance': perm.importances_mean
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feature_importance_path = os.path.join(OUTPUT_DIR, 'taskB_feature_importance.csv')
feature_importance.to_csv(feature_importance_path, index=False)

top_features = feature_importance.head(12)

plt.figure(figsize=(9, 6))
sns.barplot(data=top_features, x='Importance', y='Feature')
plt.title('Task B Top Feature Importance (Permutation Importance)')
plt.tight_layout()
feature_plot_path = os.path.join(OUTPUT_DIR, 'taskB_feature_importance.png')
plt.savefig(feature_plot_path, dpi=300, bbox_inches='tight')
plt.show()


## 7) Minimal-subset study on Task B

In [ ]:

top_k_values = [3, 5, 10, 15]
subset_rows = []

for k in top_k_values:
    selected = feature_importance.head(k)['Feature'].tolist()

    X_train_k = X_train_B[selected].copy()
    X_test_k = X_test_B[selected].copy()

    pre_k = make_preprocessor(X_train_k)
    rf_k = Pipeline([
        ('pre', pre_k),
        ('clf', RandomForestClassifier(
            n_estimators=150,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

    rf_k.fit(X_train_k, y_train_B)
    preds_k = rf_k.predict(X_test_k)
    probas_k = rf_k.predict_proba(X_test_k)[:, 1]

    subset_rows.append({
        'Top_K_Features': k,
        'Accuracy': accuracy_score(y_test_B, preds_k),
        'Precision': precision_score(y_test_B, preds_k, zero_division=0),
        'Recall': recall_score(y_test_B, preds_k, zero_division=0),
        'F1': f1_score(y_test_B, preds_k, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test_B, probas_k)
    })

subset_df = pd.DataFrame(subset_rows)
subset_path = os.path.join(OUTPUT_DIR, 'taskB_minimal_subset_results.csv')
subset_df.to_csv(subset_path, index=False)

plt.figure(figsize=(7, 5))
plt.plot(subset_df['Top_K_Features'], subset_df['F1'], marker='o')
plt.title('Task B Minimal-Subset Study')
plt.xlabel('Number of Top Features Kept')
plt.ylabel('F1 Score')
plt.xticks(top_k_values)
plt.tight_layout()
subset_plot_path = os.path.join(OUTPUT_DIR, 'taskB_minimal_subset_plot.png')
plt.savefig(subset_plot_path, dpi=300, bbox_inches='tight')
plt.show()

display(subset_df)


## 8) Optional ROC curves if prediction files were uploaded

In [ ]:

def try_load_csv(filename):
    return pd.read_csv(filename) if os.path.exists(filename) else None

taskB_pred_files = {
    'Logistic Regression': 'taskB_preds_logistic_regression.csv',
    'KNN': 'taskB_preds_knn.csv',
    'SVM': 'taskB_preds_svm.csv',
    'Random Forest': 'taskB_preds_random_forest.csv',
    'FFNN': 'taskB_preds_ffnn.csv'
}

available_roc = {}
for model_name, filename in taskB_pred_files.items():
    if os.path.exists(filename):
        df_pred = pd.read_csv(filename)
        if 'y_prob' in df_pred.columns:
            available_roc[model_name] = df_pred

if len(available_roc) > 0:
    plt.figure(figsize=(7, 6))
    for model_name, df_pred in available_roc.items():
        fpr, tpr, _ = roc_curve(df_pred['y_true'], df_pred['y_prob'])
        auc_val = roc_auc_score(df_pred['y_true'], df_pred['y_prob'])
        plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc_val:.3f})')

    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Task B ROC Curves')
    plt.legend()
    plt.tight_layout()
    roc_plot_path = os.path.join(OUTPUT_DIR, 'taskB_roc_curves.png')
    plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print("ROC plot saved.")
else:
    print("No prediction probability files were uploaded, so ROC plot was skipped.")


## 9) Auto-download one zip with all analysis outputs

## 9) Confusion matrix and classification report for Task B FFNN

These are the standard diagnostic outputs from the course Logistic Regression notebook, applied here to our best Task B model.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# Load the FFNN Task B predictions
ffnn_pred_df = pd.read_csv('taskB_preds_ffnn.csv')

y_true = ffnn_pred_df['y_true']
y_pred = ffnn_pred_df['y_pred']

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Team2 Win', 'Team1 Win'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix, Task B FFNN')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'taskB_ffnn_confusion_matrix.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.show()

# Classification report
print("\nClassification Report, Task B FFNN:")
print(classification_report(y_true, y_pred, target_names=['Team2 Win', 'Team1 Win']))


In [ ]:

import zipfile
from google.colab import files

zip_path = '/content/analysis_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        full_path = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(full_path):
            zf.write(full_path, arcname=fname)

print("Downloading:", zip_path)
files.download(zip_path)
